## Setup & Imports

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# Check versions
print(f"TensorFlow: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## Load & inspect dataset

In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Load raw data (no preprocessing yet — just exploration)
train_data = image_dataset_from_directory(
    "../data/101_food_classes_10_percent/train",  # Adjust path
    label_mode="categorical",
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
)

test_data = image_dataset_from_directory(
    "../data/101_food_classes_10_percent/test",
    label_mode="categorical",
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

print(f"Training classes: {len(train_data.class_names)}")
print(f"Training batches: {len(train_data)}")
print(f"Test batches: {len(test_data)}")
print(f"Class names (first 10): {train_data.class_names[:10]}")

## Visualize Class Distribution

In [ ]:
# Count images per class
class_counts = {}
for _, labels in train_data:
    for label in labels.numpy().argmax(axis=1):
        class_name = train_data.class_names[label]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1

# Plot
plt.figure(figsize=(16, 6))
plt.bar(range(len(class_counts)), list(class_counts.values()))
plt.title("Training Images per Class (10% Subset)")
plt.xlabel("Class Index")
plt.ylabel("Number of Images")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../images/class_distribution.png", dpi=200)
plt.show()

## Show Sample Images from Each Class

In [ ]:
def show_sample_images(dataset, class_names, num_classes=10):
    plt.figure(figsize=(16, 10))
    
    images_shown = {name: False for name in class_names}
    count = 0
    
    for images, labels in dataset:
        for i in range(len(images)):
            label_idx = labels[i].numpy().argmax()
            class_name = class_names[label_idx]
            
            if not images_shown[class_name] and count < num_classes:
                plt.subplot(2, 5, count + 1)
                plt.imshow(images[i].numpy().astype("uint8"))
                plt.title(class_name, fontsize=10)
                plt.axis("off")
                images_shown[class_name] = True
                count += 1
        
        if count >= num_classes:
            break
    
    plt.tight_layout()
    plt.savefig("../images/sample_foods.png", dpi=200)
    plt.show()

show_sample_images(train_data, train_data.class_names, num_classes=10)

## Check images shapes & Pixel range 

In [ ]:
for images, labels in train_data.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Pixel range: {images.numpy().min()} to {images.numpy().max()}")
    print(f"Label shape: {labels.shape}")
    print(f"Sample label (one-hot): {labels[0].numpy()[:10]}...")
    print(f"Label sum: {labels[0].numpy().sum()}")

## Baseline — Random Guessing Accuracy

In [ ]:
# Theoretical random guessing accuracy
random_acc = 1 / len(train_data.class_names)
print(f"Random guessing accuracy: {random_acc:.4f} ({random_acc*100:.2f}%)")
print(f"Your model should do much better than this!")

## Why Transfer Learning?

In [ ]:
# Show why we use transfer learning instead of training from scratch
total_params_efficientnet = 7_200_000  # EfficientNetV2B0 params
trainable_params_our_model = 128_000    # Our dense head

print(f"EfficientNetV2B0 total parameters: {total_params_efficientnet:,}")
print(f"Our trainable head parameters: {trainable_params_our_model:,}")
print(f"\nWithout transfer learning, we'd need to train {total_params_efficientnet:,} params")
print(f"With transfer learning, we only train {trainable_params_our_model:,} params")
print(f"That's {total_params_efficientnet / trainable_params_our_model:.0f}x fewer parameters!")